# EAG model check in Google Colab

Runs the Evidence-Aggregation-Growth model on your own data, or on built-in example data, without installing anything on your computer.

1. Run all cells (Runtime, Run all).
2. To analyse your own data, set `USE_EXAMPLE = False` in the second code cell and upload your three tables when asked.

In [ ]:
REPO = "https://github.com/lethanhhuydn/eag-model.git"   # address of the repository
!pip -q install "git+{REPO}"

In [ ]:
import pandas as pd
from eagmodel import EAGData, SimConfig, fit, simulate
from eagmodel.validate import validate
from eagmodel.report import to_html

USE_EXAMPLE = True      # set to False to upload your own files
THRESHOLD = 0.6         # readiness threshold on the competence scale, or None

if USE_EXAMPLE:
    data, _ = simulate(SimConfig(n_students=200), seed=2026)
else:
    from google.colab import files
    print('Upload analytic.csv, holistic.csv and (optionally) students.csv')
    up = files.upload()
    read = lambda name: pd.read_excel(name) if name.endswith('.xlsx') else pd.read_csv(name)
    pick = lambda key: next((n for n in up if key in n.lower()), None)
    data = EAGData.from_frames(read(pick('analytic')), read(pick('holistic')),
                               read(pick('students')) if pick('students') else None)
print(data.n_students, 'students,', data.n_components, 'components,', data.n_occasions, 'occasion(s)')

In [ ]:
model = fit(data, threshold=THRESHOLD, extra_models=True, verbose=True)
report = validate(model, threshold=THRESHOLD)
print(report.headline)
report.checks_frame()

In [ ]:
from IPython.display import HTML
html = to_html(model, report)
open('eag_report.html', 'w', encoding='utf-8').write(html)
report.tables['students'].to_csv('students.csv', index=False)
HTML(html)

In [ ]:
# download the report and the student table
try:
    from google.colab import files
    files.download('eag_report.html'); files.download('students.csv')
except ImportError:
    print('Saved eag_report.html and students.csv in the working folder')